# 07｜Web 操作介面

定義影片預覽、偵測參數、處理控制與 LINE 通知設定頁面。


In [ ]:
HTML_PAGE = """
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<title>溺水偵測系統</title>
<style>
    body { font-family: -apple-system, "Segoe UI", Arial, sans-serif; background:#0f1116; color:#e6e6e6; margin:0; padding:20px; }
    h1 { font-size: 22px; margin-bottom: 4px; }
    .subtitle { color:#9aa0a6; margin-bottom: 20px; font-size: 13px; }
    .container { display:flex; gap: 20px; }
    .left { flex: 2; }
    .right { flex: 1; background:#1a1d24; border-radius:10px; padding:16px; }
    #videoImg { width:100%; border-radius:10px; background:#000; display:block; }
    .warning-banner { display:none; background:#d32f2f; color:white; text-align:center;
                       padding:10px; font-weight:bold; font-size:18px; border-radius:6px; margin-top:8px; }
    .warning-banner.show { display:block; animation: pulse 1s infinite; }
    @keyframes pulse { 0%{opacity:1;} 50%{opacity:0.6;} 100%{opacity:1;} }
    #statsBox { background:#1a1d24; border-radius:10px; padding:14px; margin-top:10px;
                white-space:pre-wrap; font-family: Consolas, monospace; font-size:13px; min-height:140px; }
    label { display:block; margin-top:14px; margin-bottom:4px; font-size:13px; color:#c9ccd1; }
    input[type=range] { width:100%; }
    input[type=text], input[type=password] { width:100%; padding:6px; border-radius:6px; border:1px solid #333;
                        background:#0f1116; color:#e6e6e6; box-sizing:border-box; }
    .radio-group label { display:inline-block; margin-right:16px; font-size:14px; }
    button { padding:10px; margin-top:10px; border:none; border-radius:8px;
             font-size:14px; cursor:pointer; font-weight:600; }
    button:disabled { opacity:0.4; cursor:not-allowed; }
    input:disabled { opacity:0.4; cursor:not-allowed; }
    .btn-row { display:flex; gap:8px; }
    .btn-row button { flex:1; }
    .btn-start { background:#4CAF50; color:white; width:100%; }
    .btn-pause { background:#FF9800; color:white; }
    .btn-stop { background:#555; color:white; }
    .btn-upload { background:#2196F3; color:white; white-space:nowrap; }
    .btn-cancel { background:#f44336; color:white; white-space:nowrap; }
    .btn-reset { background:#607D8B; color:white; width:100%; }
    .btn-download { background:#009688; color:white; width:100%; text-decoration:none;
                     text-align:center; display:block; box-sizing:border-box; }
    .btn-test-line { background:#06C755; color:white; width:100%; }
    .slider-val { float:right; color:#8ab4f8; }
    .section-title { font-weight:700; margin-top:4px; }
    .hint { color:#9aa0a6; font-size:11px; margin-top:6px; }

    .file-row { display:flex; gap:8px; align-items:center; }
    .file-row input[type=file] { flex:1; padding:6px; border-radius:6px; border:1px solid #333;
                                  background:#0f1116; color:#e6e6e6; box-sizing:border-box; }

    .upload-status { margin-top:8px; font-size:12px; color:#8ab4f8; min-height:16px; }
    .upload-status.success { color:#4CAF50; }
    .upload-status.error { color:#f44336; }
    .progress-bar-bg { width:100%; background:#333; border-radius:6px; height:8px; margin-top:4px; overflow:hidden; display:none; }
    .progress-bar-fill { height:100%; background:#4CAF50; width:0%; transition: width 0.15s; }

    .locked-hint { display:none; color:#FF9800; font-size:12px; margin-top:8px; padding:8px;
                   background:rgba(255,152,0,0.1); border-radius:6px; }
    .locked-hint.show { display:block; }

    .line-section { border-top:1px solid #333; margin-top:16px; padding-top:12px; }
    .line-status { font-size:12px; margin-top:6px; min-height:16px; }
    .checkbox-row { display:flex; align-items:center; gap:8px; margin-top:10px; }
    .checkbox-row label { margin:0; }
    .checkbox-row input[type=checkbox] { width:16px; height:16px; }
</style>
</head>
<body>

<h1>🏊 溺水偵測系統</h1>
<div class="subtitle">Flask + MJPEG 串流版</div>

<div class="container">
    <div class="left">
        <img id="videoImg" src="/video_feed">
        <div id="warningBanner" class="warning-banner">⚠️ POSSIBLE DROWNING ⚠️</div>
        <div id="statsBox">尚未開始處理</div>
        <a class="btn-download" id="downloadBtn" href="/download_log" style="display:none;">⬇ 下載偵測紀錄 (CSV)</a>
    </div>

    <div class="right">
        <div class="section-title">影片來源</div>
        <div class="radio-group">
            <label><input type="radio" name="source_type" value="upload" checked onchange="toggleSource()" class="ctrl-input"> 上傳影片</label>
            <label><input type="radio" name="source_type" value="stream" onchange="toggleSource()" class="ctrl-input"> 視訊串流</label>
        </div>

        <div id="uploadBlock">
            <label>選擇影片檔案</label>
            <div class="file-row">
                <input type="file" id="videoFile" accept="video/*" class="ctrl-input" onchange="onFileSelected()">
                <button class="btn-upload ctrl-input" id="uploadBtn" onclick="uploadFile()" disabled>⬆ 上傳</button>
                <button class="btn-cancel" id="cancelBtn" onclick="cancelUpload()" style="display:none;">✕ 取消</button>
            </div>
            <div class="progress-bar-bg" id="progressBarBg">
                <div class="progress-bar-fill" id="progressBarFill"></div>
            </div>
            <div class="upload-status" id="uploadStatus"></div>
        </div>

        <div id="streamBlock" style="display:none;">
            <label>攝影機編號 或 串流網址</label>
            <input type="text" id="streamUrl" class="ctrl-input"
                   placeholder="例如 0（本機攝影機）或 rtsp://192.168.1.100:554/stream">
            <div class="hint">
                純數字（如 0、1）＝讀取本機攝影機裝置編號（僅限本機執行有效，Colab雲端環境沒有實體攝影機）<br>
                RTSP/HTTP網址＝連接遠端串流來源（目前尚未實測）
            </div>
        </div>

        <label>信心門檻 (conf_threshold) <span class="slider-val" id="confVal">0.25</span></label>
        <input type="range" id="confSlider" class="ctrl-input" min="0.05" max="0.95" step="0.01" value="0.25"
               oninput="document.getElementById('confVal').innerText=this.value">

        <label>Drowning 佔比門檻 <span class="slider-val" id="ratioVal">0.60</span></label>
        <input type="range" id="ratioSlider" class="ctrl-input" min="0.1" max="1.0" step="0.05" value="0.6"
               oninput="document.getElementById('ratioVal').innerText=this.value">

        <label>觀察視窗秒數 <span class="slider-val" id="windowVal">5.0</span></label>
        <input type="range" id="windowSlider" class="ctrl-input" min="1" max="10" step="0.5" value="5"
               oninput="document.getElementById('windowVal').innerText=this.value">

        <label>最低樣本比例 <span class="slider-val" id="minRatioVal">0.30</span></label>
        <input type="range" id="minRatioSlider" class="ctrl-input" min="0.05" max="1.0" step="0.05" value="0.3"
               oninput="document.getElementById('minRatioVal').innerText=this.value">

        <button class="btn-reset ctrl-input" id="resetBtn" onclick="resetDefaults()">↺ 回復預設值</button>

        <div class="line-section">
            <div class="section-title">LINE 通知（純文字）</div>
            <div class="checkbox-row">
                <input type="checkbox" id="lineEnabled" class="ctrl-input" onchange="toggleLineFields()">
                <label style="margin:0;">啟用溺水通知（透過LINE官方帳號廣播文字訊息）</label>
            </div>

            <div id="lineFields" style="display:none;">
                <label>CHANNEL_ACCESS_TOKEN</label>
                <input type="password" id="lineToken" class="ctrl-input" placeholder="貼上你的LINE Channel Access Token">
                <div class="hint">
                    通知內容包含：時間（影片時間或系統時間）、幀編號、Tracking ID，僅發送文字，不含截圖。
                </div>

                <button class="btn-test-line ctrl-input" onclick="testLine()">🔔 測試發送</button>
                <div class="line-status" id="lineTestStatus"></div>
            </div>
        </div>

        <div class="locked-hint" id="lockedHint">🔒 處理中，參數已鎖定。按「停止」後才能調整並重新開始。</div>

        <div class="hint">※ 參數需在按下「開始處理」前設定好</div>

        <button class="btn-start" id="startBtn" onclick="startProcessing()">▶ 開始處理</button>
        <div class="btn-row">
            <button class="btn-pause" id="pauseBtn" onclick="togglePause()" disabled>⏸ 暫停</button>
            <button class="btn-stop" id="stopBtn" onclick="stopProcessing()" disabled>⏹ 停止</button>
        </div>
    </div>
</div>

<script>
let isPaused = false;
let isProcessing = false;
let currentXhr = null;

const DEFAULTS = { conf: 0.25, ratio: 0.6, window: 5.0, minRatio: 0.3 };

function resetDefaults() {
    document.getElementById('confSlider').value = DEFAULTS.conf;
    document.getElementById('confVal').innerText = DEFAULTS.conf.toFixed(2);
    document.getElementById('ratioSlider').value = DEFAULTS.ratio;
    document.getElementById('ratioVal').innerText = DEFAULTS.ratio.toFixed(2);
    document.getElementById('windowSlider').value = DEFAULTS.window;
    document.getElementById('windowVal').innerText = DEFAULTS.window.toFixed(1);
    document.getElementById('minRatioSlider').value = DEFAULTS.minRatio;
    document.getElementById('minRatioVal').innerText = DEFAULTS.minRatio.toFixed(2);
}

function toggleSource() {
    const type = document.querySelector('input[name=source_type]:checked').value;
    document.getElementById('uploadBlock').style.display = (type === 'upload') ? 'block' : 'none';
    document.getElementById('streamBlock').style.display = (type === 'stream') ? 'block' : 'none';
}

function toggleLineFields() {
    const enabled = document.getElementById('lineEnabled').checked;
    document.getElementById('lineFields').style.display = enabled ? 'block' : 'none';
}

function testLine() {
    const token = document.getElementById('lineToken').value.trim();
    if (!token) { alert('請先輸入CHANNEL_ACCESS_TOKEN'); return; }

    const statusEl = document.getElementById('lineTestStatus');
    statusEl.style.color = '#8ab4f8';
    statusEl.innerText = '發送中...';

    const formData = new FormData();
    formData.append('line_token', token);

    fetch('/test_line', { method: 'POST', body: formData })
        .then(r => r.json())
        .then(data => {
            statusEl.style.color = data.ok ? '#4CAF50' : '#f44336';
            statusEl.innerText = (data.ok ? '✅ ' : '❌ ') + data.msg;
        });
}

function onFileSelected() {
    const fileInput = document.getElementById('videoFile');
    const uploadBtn = document.getElementById('uploadBtn');
    uploadBtn.disabled = (fileInput.files.length === 0);
    document.getElementById('uploadStatus').innerText = '';
    document.getElementById('uploadStatus').className = 'upload-status';
    document.getElementById('progressBarBg').style.display = 'none';
    document.getElementById('progressBarFill').style.width = '0%';
}

function uploadFile() {
    const fileInput = document.getElementById('videoFile');
    if (fileInput.files.length === 0) { alert('請先選擇影片檔案'); return; }

    const formData = new FormData();
    formData.append('video_file', fileInput.files[0]);

    const xhr = new XMLHttpRequest();
    currentXhr = xhr;

    const uploadBtn = document.getElementById('uploadBtn');
    const cancelBtn = document.getElementById('cancelBtn');
    const progressBarBg = document.getElementById('progressBarBg');
    const progressBarFill = document.getElementById('progressBarFill');
    const uploadStatus = document.getElementById('uploadStatus');

    uploadBtn.disabled = true;
    cancelBtn.style.display = 'inline-block';
    progressBarBg.style.display = 'block';
    progressBarFill.style.width = '0%';
    uploadStatus.className = 'upload-status';
    uploadStatus.innerText = '準備上傳...';

    xhr.upload.onprogress = function(e) {
        if (e.lengthComputable) {
            const pct = Math.round((e.loaded / e.total) * 100);
            progressBarFill.style.width = pct + '%';
            uploadStatus.innerText = `上傳中... ${pct}%`;
        }
    };

    xhr.onload = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        if (xhr.status === 200) {
            const data = JSON.parse(xhr.responseText);
            if (data.ok) {
                uploadStatus.className = 'upload-status success';
                uploadStatus.innerText = '✅ 上傳完成，已顯示首幀，可以按「開始處理」';
                progressBarFill.style.width = '100%';
                document.getElementById('videoImg').src = '/video_feed?t=' + Date.now();
                document.getElementById('downloadBtn').style.display = 'none';
                uploadBtn.disabled = true;
            } else {
                uploadStatus.className = 'upload-status error';
                uploadStatus.innerText = '❌ ' + data.msg;
                alert(data.msg);
                uploadBtn.disabled = false;
            }
        } else {
            uploadStatus.className = 'upload-status error';
            uploadStatus.innerText = '❌ 上傳失敗';
            uploadBtn.disabled = false;
        }
    };

    xhr.onerror = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        uploadStatus.className = 'upload-status error';
        uploadStatus.innerText = '❌ 網路錯誤，上傳失敗';
        uploadBtn.disabled = false;
    };

    xhr.onabort = function() {
        cancelBtn.style.display = 'none';
        currentXhr = null;
        uploadStatus.className = 'upload-status';
        uploadStatus.innerText = '已取消上傳';
        progressBarFill.style.width = '0%';
        progressBarBg.style.display = 'none';
        uploadBtn.disabled = false;
    };

    xhr.open('POST', '/upload', true);
    xhr.send(formData);
}

function cancelUpload() {
    if (currentXhr) {
        currentXhr.abort();
    }
}

function startProcessing() {
    const sourceType = document.querySelector('input[name=source_type]:checked').value;
    const formData = new FormData();
    formData.append('source_type', sourceType);
    formData.append('conf_threshold', document.getElementById('confSlider').value);
    formData.append('warning_ratio', document.getElementById('ratioSlider').value);
    formData.append('warning_window_sec', document.getElementById('windowSlider').value);
    formData.append('min_ratio_of_window', document.getElementById('minRatioSlider').value);

    const lineEnabled = document.getElementById('lineEnabled').checked;
    formData.append('line_enabled', lineEnabled ? 'true' : 'false');
    formData.append('line_token', document.getElementById('lineToken').value.trim());

    if (sourceType === 'stream') {
        formData.append('stream_url', document.getElementById('streamUrl').value);
    }

    document.getElementById('downloadBtn').style.display = 'none';

    fetch('/start', { method: 'POST', body: formData })
        .then(r => r.json())
        .then(data => {
            if (data.ok) {
                setLocked(true);
            } else {
                alert(data.msg);
            }
        });
}

function stopProcessing() {
    fetch('/stop', { method: 'POST' })
        .then(r => r.json())
        .then(data => {
            console.log(data.msg);
            isPaused = false;
            document.getElementById('pauseBtn').innerText = '⏸ 暫停';
            setLocked(false);
        });
}

function togglePause() {
    const endpoint = isPaused ? '/resume' : '/pause';
    fetch(endpoint, { method: 'POST' })
        .then(r => r.json())
        .then(data => {
            if (data.ok) {
                isPaused = !isPaused;
                document.getElementById('pauseBtn').innerText = isPaused ? '▶ 繼續' : '⏸ 暫停';
            } else {
                alert(data.msg);
            }
        });
}

function setLocked(locked) {
    isProcessing = locked;
    document.querySelectorAll('.ctrl-input').forEach(el => { el.disabled = locked; });
    document.getElementById('startBtn').disabled = locked;
    document.getElementById('lockedHint').classList.toggle('show', locked);
    document.getElementById('pauseBtn').disabled = !locked;
    document.getElementById('stopBtn').disabled = !locked;

    if (locked) {
        document.getElementById('uploadBtn').disabled = true;
    }
}

setInterval(() => {
    fetch('/stats')
        .then(r => r.json())
        .then(data => {
            document.getElementById('statsBox').innerText = data.stats;
            const banner = document.getElementById('warningBanner');
            if (data.warning) { banner.classList.add('show'); }
            else { banner.classList.remove('show'); }

            isPaused = data.paused;
            document.getElementById('pauseBtn').innerText = isPaused ? '▶ 繼續' : '⏸ 暫停';

            if (data.running !== isProcessing) {
                setLocked(data.running);
            }

            document.getElementById('downloadBtn').style.display = data.log_ready ? 'block' : 'none';
        });
}, 1000);

setLocked(false);
</script>

</body>
</html>
"""


In [ ]:
print("Web 操作介面已載入。")
